<a href="https://colab.research.google.com/github/HemanthSagarSR/BIS_LAB/blob/main/week6_parallel_cellular_algo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# --- 1. PROBLEM SPECIFICATION ---
# Optimization objective: Find the allocation of Tellers that minimizes wait times.
# Cell State vector: [Standard_Tellers, Loan_Tellers]
MAX_TELLERS = 15

def evaluate_fitness(strategy):
    """
    Objective Function: Simulates/estimates wait times based on teller allocation.
    Lower score = better configuration (lower customer wait times).
    """
    std_tellers, loan_tellers = strategy

    # Total tellers constraint penalty
    if (std_tellers + loan_tellers) > MAX_TELLERS or std_tellers < 1 or loan_tellers < 1:
        return 9999.0  # Invalid configuration penalty

    # Simplified queuing model: wait time is inversely proportional to teller count
    # (Assuming fixed arrival rates: standard traffic is higher than loan traffic)
    std_wait = 50.0 / std_tellers
    loan_wait = 120.0 / loan_tellers

    # Total cost = Average wait time + operational cost of keeping tellers active
    total_wait_time = std_wait + loan_wait
    staff_cost = (std_tellers + loan_tellers) * 0.5

    return total_wait_time + staff_cost

# --- 2. INITIALIZATION ---
grid_size = 5  # 5x5 grid = 25 parallel solution candidates
generations = 20
alpha = 0.5   # Adaptation rate (from the formula)

# Initialize a grid of strategies randomly
# State: [standard_tellers, loan_tellers]
grid = np.random.randint(1, 8, size=(grid_size, grid_size, 2)).astype(float)
fitness_grid = np.zeros((grid_size, grid_size))

# --- 3. HELPER FOR MOORE NEIGHBORHOOD (8-NEIGHBOR) ---
def get_best_neighbor(grid, r, c):
    """Finds the best performing strategy among immediate 8 neighbors."""
    best_fitness = float('inf')
    best_strategy = None

    # Loop through Moore neighborhood offsets (-1, 0, 1)
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            nr, nc = r + dr, c + dc
            # Toroidal boundary conditions (grid wraps around edges)
            nr = nr % grid_size
            nc = nc % grid_size

            neighbor_strategy = grid[nr, nc]
            neighbor_fitness = evaluate_fitness(neighbor_strategy)

            if neighbor_fitness < best_fitness:
                best_fitness = neighbor_fitness
                best_strategy = neighbor_strategy

    return best_strategy

# --- 4. MAIN PCA WORKING PROCESS LOOP ---
print("Starting Parallel Cellular Algorithm Optimization...\n")

for gen in range(generations):
    # Step 4: Evaluate Fitness for all cells
    for r in range(grid_size):
        for c in range(grid_size):
            fitness_grid[r, c] = evaluate_fitness(grid[r, c])

    # Track the global best for reporting
    best_idx = np.unravel_index(np.argmin(fitness_grid), fitness_grid.shape)
    best_gen_strategy = np.round(grid[best_idx])
    best_gen_fitness = fitness_grid[best_idx]

    print(f"Gen {gen+1:02d} | Best Strategy Found: Tellers (Std: {int(best_gen_strategy[0])}, Loan: {int(best_gen_strategy[1])}) | Wait Score: {best_gen_fitness:.2f}")

    # Step 5: Update States in parallel (simulated via temporary grid copy)
    next_grid = np.copy(grid)
    for r in range(grid_size):
        for c in range(grid_size):
            x_best_N = get_best_neighbor(grid, r, c)
            x_current = grid[r, c]

            # Applying PCA Update Rule formula: x_new = x_current + alpha * (x_best_N - x_current)
            updated_strategy = x_current + alpha * (x_best_N - x_current)

            # Mutate slightly to add explore factors, keep boundaries whole numbers
            updated_strategy += np.random.uniform(-0.5, 0.5, size=2)
            next_grid[r, c] = np.clip(np.round(updated_strategy), 1, MAX_TELLERS)

    grid = next_grid

# --- 5. FINAL EXTRACED RESULT ---
print("\n" + "="*50)
print(f"OPTIMAL SMART QUEUE STRATEGY DEPLOYED:")
print(f"Allocated Counters for Standard Transactions: {int(best_gen_strategy[0])}")
print(f"Allocated Counters for Loan/Complex Transactions: {int(best_gen_strategy[1])}")
print(f"Minimum Projected Performance Cost Index: {best_gen_fitness:.2f}")
print("="*50)


Starting Parallel Cellular Algorithm Optimization...

Gen 01 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 02 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 03 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 04 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 05 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 06 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 07 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 08 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 09 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 10 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 11 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 12 | Best Strategy Found: Tellers (Std: 7, Loan: 6) | Wait Score: 33.64
Gen 13 | Best Strategy Found: Tell